In [0]:
%pip install kagglehub --quiet
%restart_python

---
# Descargar el dataset crudo con kagglehub

In [0]:
import os
import shutil
import kagglehub

# 1. Resolver la ruta a la carpeta data/ del repositorio
repo_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
data_dir = os.path.join(repo_path, "data")
os.makedirs(data_dir, exist_ok=True)

# 2. Descarga automática desde Kaggle (a la caché local)
cached_path = kagglehub.dataset_download("uciml/iris")

# 3. Copia a /data/iris.csv dentro del repositorio
origen_csv = os.path.join(cached_path, "Iris.csv")
destino_csv = os.path.join(data_dir, "iris.csv")
shutil.copyfile(origen_csv, destino_csv)

print(f"Dataset disponible en: {destino_csv}")

---
# Config-Driven

In [0]:
# 1. Definición de Widgets interactivos (Config-Driven)
dbutils.widgets.text("entorno", "dev", "Entorno (dev/prod)")
dbutils.widgets.text("ruta_csv", destino_csv, "Ruta absoluta del dataset CSV")

# 2. Captura de los valores configurados
env = dbutils.widgets.get("entorno")
input_csv_path = dbutils.widgets.get("ruta_csv")

# 3. Definición de rutas Delta Lake para la Arquitectura Medallón
base_lakehouse = f"dbfs:/lakehouse/{env}/iris"
bronze_path = f"{base_lakehouse}/bronze"
silver_path = f"{base_lakehouse}/silver"
gold_path = f"{base_lakehouse}/gold"

print(f"Entorno activo: {env.upper()}")
print(f"Dataset de entrada (Config): {input_csv_path}")
print(f"Capa Bronce destino: {bronze_path}")

# Capa Bronce

In [0]:
import os
from pyspark.sql.functions import col, current_timestamp

# 1. Parámetros de ejecución (Config-Driven)
try:
    env = dbutils.widgets.get("entorno")
except Exception:
    env = "dev"

try:
    input_csv_path = dbutils.widgets.get("ruta_csv")
except Exception:
    repo_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
    input_csv_path = os.path.join(repo_path, "data", "iris.csv")

# 2. Nombres de tablas en Unity Catalog (catalogo.esquema.tabla)
catalog_name = "workspace"
schema_name = "default"

tabla_bronce = f"{catalog_name}.{schema_name}.bronze_iris_{env}"
tabla_silver = f"{catalog_name}.{schema_name}.silver_iris_{env}"
tabla_gold = f"{catalog_name}.{schema_name}.gold_iris_{env}"

print(f"-> Destino de Capa Bronce: {tabla_bronce}")

# 3. Lectura distribuida del CSV con PySpark (Etapa 2)
df_raw = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(input_csv_path)
)

# 4. Metadatos técnicos de auditoría adaptados a Unity Catalog
df_bronze = (
    df_raw
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source_file", col("_metadata.file_name"))
)

# 5. Persistencia como Managed Table en Delta Lake (Etapa 4)
(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(tabla_bronce)
)

# 6. Validación de lectura desde Delta Lake
df_validar = spark.table(tabla_bronce)
display(df_validar.limit(5))